# 01 Load and QC Datalogger Data

This notebook loads ESP8266 datalogger measurement CSV files from **2026-05-29 onward only**. It preserves duplicated sensor columns from the two I2C buses by renaming repeated headers with `__bus0`, `__bus1`, ... suffixes.

In [ ]:
from pathlib import Path
import csv
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

DATA_START_DATE = "20260529"
EXPECTED_INTERVAL_SECONDS = 20
DATA_DIR = Path("..") / "test_data"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

print("pandas", pd.__version__)
print("data directory", DATA_DIR.resolve())

In [ ]:
def data_file_date(path: Path) -> str:
    match = re.fullmatch(r"data_(\d{8})\.csv", path.name)
    return match.group(1) if match else ""


def measurement_files(data_dir: Path = DATA_DIR, start_date: str = DATA_START_DATE) -> list[Path]:
    files = []
    for path in sorted(data_dir.glob("data_*.csv")):
        date = data_file_date(path)
        if date and date >= start_date:
            files.append(path)
    return files


def unique_headers(headers: list[str]) -> list[str]:
    total = Counter(headers)
    seen = defaultdict(int)
    out = []
    for header in headers:
        if total[header] == 1:
            out.append(header)
        else:
            bus = seen[header]
            out.append(f"{header}__bus{bus}")
            seen[header] += 1
    return out


def inspect_row_widths(path: Path) -> tuple[list[str], pd.DataFrame]:
    issues = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        raw_headers = next(reader)
        expected_width = len(raw_headers)
        for line_number, row in enumerate(reader, start=2):
            if len(row) != expected_width:
                issues.append({
                    "source_file": path.name,
                    "line_number": line_number,
                    "expected_columns": expected_width,
                    "actual_columns": len(row),
                })
    return raw_headers, pd.DataFrame(issues)


def read_measurement_file(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw_headers, width_issues = inspect_row_widths(path)
    names = unique_headers(raw_headers)
    df = pd.read_csv(
        path,
        names=names,
        skiprows=1,
        na_values=["", "nan", "NaN", "NAN"],
        keep_default_na=True,
        on_bad_lines="skip",
    )
    df.insert(0, "source_file", path.name)
    df.insert(1, "source_date", data_file_date(path))
    return df, width_issues


files = measurement_files()
assert files, "No measurement files found from the configured start date."
print("Selected measurement files:")
for file in files:
    print(" -", file.name)

assert all(data_file_date(file) >= DATA_START_DATE for file in files)

In [ ]:
frames = []
width_issue_frames = []
for file in files:
    frame, issues = read_measurement_file(file)
    frames.append(frame)
    width_issue_frames.append(issues)

data = pd.concat(frames, ignore_index=True)
row_width_issues = pd.concat(width_issue_frames, ignore_index=True) if width_issue_frames else pd.DataFrame()

data["timestamp_utc"] = pd.to_datetime(data["timestamp_utc"], errors="coerce", utc=False)
data["file_date"] = pd.to_datetime(data["source_date"], format="%Y%m%d", errors="coerce").dt.date
data["date"] = data["timestamp_utc"].dt.date
data["hour"] = data["timestamp_utc"].dt.hour

print("Rows loaded:", len(data))
print("Columns loaded:", len(data.columns))
print("Timestamp parse failures:", int(data["timestamp_utc"].isna().sum()))
print("Row width issues:", len(row_width_issues))

data.head()

In [ ]:
def compute_gaps(df: pd.DataFrame, expected_interval_s: int = EXPECTED_INTERVAL_SECONDS) -> pd.DataFrame:
    pieces = []
    valid = df.dropna(subset=["timestamp_utc"]).sort_values(["source_file", "timestamp_utc"])
    for source_file, group in valid.groupby("source_file", sort=True):
        ts = group["timestamp_utc"].reset_index(drop=True)
        deltas = ts.diff().dt.total_seconds()
        gap = pd.DataFrame({
            "source_file": source_file,
            "previous_timestamp": ts.shift(1),
            "timestamp_utc": ts,
            "delta_seconds": deltas,
        })
        pieces.append(gap)
    out = pd.concat(pieces, ignore_index=True)
    out["missing_samples_est"] = np.maximum(
        0,
        np.floor(out["delta_seconds"].fillna(expected_interval_s) / expected_interval_s).astype(int) - 1,
    )
    return out


gaps_all = compute_gaps(data)
large_gaps = gaps_all[gaps_all["delta_seconds"] > 120].copy()
large_gaps["gap_minutes"] = large_gaps["delta_seconds"] / 60

duplicate_timestamps = (
    data.dropna(subset=["timestamp_utc"])
    .groupby(["source_file", "timestamp_utc"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)

file_summary = (
    data.groupby("source_file", as_index=False)
    .agg(
        rows=("timestamp_utc", "size"),
        parseable_timestamps=("timestamp_utc", "count"),
        start_utc=("timestamp_utc", "min"),
        end_utc=("timestamp_utc", "max"),
        gps_sat_median=("nb_sat", "median"),
        hdop_median=("HDOP", "median"),
    )
)
gap_summary = large_gaps.groupby("source_file", as_index=False).agg(
    gaps_gt_120s=("delta_seconds", "size"),
    max_gap_s=("delta_seconds", "max"),
    missing_samples_est=("missing_samples_est", "sum"),
)
file_summary = file_summary.merge(gap_summary, on="source_file", how="left").fillna({
    "gaps_gt_120s": 0,
    "max_gap_s": 0,
    "missing_samples_est": 0,
})

file_summary

In [ ]:
print("Large gaps over 120 seconds:")
display(large_gaps[["source_file", "previous_timestamp", "timestamp_utc", "delta_seconds", "gap_minutes", "missing_samples_est"]])

print("Duplicate timestamps:", len(duplicate_timestamps))
display(duplicate_timestamps.head(20))

print("Row width issues:", len(row_width_issues))
display(row_width_issues.head(20))

In [ ]:
hourly_base = data.dropna(subset=["timestamp_utc"]).assign(
    date=lambda x: x["timestamp_utc"].dt.strftime("%Y-%m-%d"),
    hour=lambda x: x["timestamp_utc"].dt.hour,
)
hourly_coverage = hourly_base.groupby(["date", "hour"], as_index=False).agg(
    rows=("timestamp_utc", "size"),
    unique_timestamps=("timestamp_utc", "nunique"),
)
hourly_coverage["duplicate_timestamp_rows"] = hourly_coverage["rows"] - hourly_coverage["unique_timestamps"]
hourly_coverage["expected_rows"] = 3600 / EXPECTED_INTERVAL_SECONDS
hourly_coverage["coverage_fraction"] = (hourly_coverage["unique_timestamps"] / hourly_coverage["expected_rows"]).clip(upper=1)

display(hourly_coverage.head(48))
display(hourly_coverage.query("date == '2026-05-31'"))